In [2]:
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
from einops import rearrange, repeat

from src.models.ViT import ViT

from src.models.HIPT import HIPT

In [56]:

batch_size = 4
img_n_dims = 3

x = torch.rand((batch_size,3,96,96,96))

block_size = 8

n_blocks = [dim//block_size for dim in x.shape[-img_n_dims:]]  # get the number of blocks in each dimension


# x_blocks = rearrange(x, 'b c (p1 b1) (p2 b2) (p3 b3) -> b c p1 p2 p3 b1 b2 b3', b1=block_size, b2=block_size, b3=block_size)
x_blocks = x.unfold(2, block_size, block_size).unfold(3, block_size, block_size).unfold(4, block_size, block_size)
print(x_blocks.shape)

x_blocks = rearrange(x_blocks, 'batch c n1 n2 n3 b1 b2 b3 -> batch (n1 n2 n3) c  b1 b2 b3')     # n1, n2, n3 are the number of blocks in each dimension
print(x_blocks.shape)

torch.Size([4, 3, 12, 12, 12, 8, 8, 8])
torch.Size([4, 1728, 3, 8, 8, 8])


In [57]:
ps = 2
ViT_1 = ViT(image_size=(block_size, block_size, block_size), patch_size=(ps,ps,ps),
                    dim=512, depth=1, 
                    heads=4, mlp_dim=256, pool = 'cls', 
                    channels = 3, dim_head = 128, emb_dropout = 0.0,
                    dropout = 0.0) 



In [58]:
ViT_1 = ViT_1.to('cuda')
#x_blocks = x_blocks.to('cuda')


features_1 = []
for mini_bs_idx in range(x_blocks.shape[1]):
    x_block = x_blocks[:, mini_bs_idx].to('cuda')
    
    temp_out = ViT_1(x_block)#.unsqueeze(1)
    features_1.append(temp_out)
    #print(temp_out.shape)

    #ViT1_out = ViT_1(x_block)
features_1 = torch.stack(features_1, dim=1)

In [61]:
n_blocks

[12, 12, 12]

In [ ]:
n1, n2, n3 = n_blocks

features_2 = rearrange(features_1, 'batch (n1 n2 n3) v  -> batch v n1 n2 n3', n1=n1, n2=n2, n3=n3)

features_2.shape

torch.Size([4, 512, 24, 24, 24])

In [66]:
tuple123 = tuple(n_blocks)

In [67]:
np.multiply(list(tuple123))

TypeError: multiply() takes from 2 to 3 positional arguments but 1 were given

In [64]:
block_size_2 = 12
ps = 2
ViT_2 = ViT(image_size=tuple(n_blocks), patch_size=(ps,ps,ps),
                    dim=128, depth=1, 
                    heads=4, mlp_dim=256, pool = 'cls', 
                    channels = 512, dim_head = 256, emb_dropout = 0.0,
                    dropout = 0.0)

In [97]:
ViT_2.to('cuda')
features_out = ViT_2(features_2)

In [99]:
features_out.shape

torch.Size([4, 128])

In [1]:
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
from einops import rearrange, repeat

from src.models.ViT import ViT

from src.models.HIPT import HIPT

x = torch.rand((16,3,96,96,96))
#x = torch.rand((4,3,96,96*2,96*2))


batch_size, channels, depth, height, width = x.shape
HIPT_model = HIPT(channels, depth, height, width)

HIPT_model.to('cuda')

print( )

In [2]:

#x = x.to('cuda')

out = HIPT_model(x)

OutOfMemoryError: CUDA out of memory. Tried to allocate 14.00 MiB. GPU 

In [31]:
del out

torch.cuda.empty_cache()